# Strong DiFuMo GAT Colab Experiment

Runs DiFuMo GAT/GATv2 graph experiments on CUDA/AMP. The DiFuMo MLP controls live in `experiments/difumo_mlp_colab.ipynb`, and artifact-only comparisons live in `experiments/difumo_compare_colab.ipynb`.

In [ ]:
import os, platform, subprocess, sys
print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('bf16:', torch.cuda.is_bf16_supported())
except Exception as exc:
    print('Torch check failed:', exc)
print('Platform:', platform.platform())

In [ ]:
# Colab dependency install. Runtime -> Change runtime type -> A100 GPU is recommended.
!pip -q install nilearn nibabel scipy huggingface-hub safetensors adapters transformers pyarrow matplotlib pandas scikit-learn tqdm umap-learn torch-geometric

In [ ]:
# Repo clone/pull for Colab. Skip this cell when running inside an existing checkout.
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = '/content/neurovlm_gnn'
if not os.path.exists(REPO_DIR):
    !git clone --branch $REPO_BRANCH --single-branch $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $REPO_BRANCH
    !git -C $REPO_DIR checkout $REPO_BRANCH
    !git -C $REPO_DIR pull --ff-only origin $REPO_BRANCH
os.chdir(REPO_DIR)
!pip -q install -e '.[viz,notebook,metrics]'
print('Working directory:', os.getcwd())

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configure Runs

Everything important is written under `DRIVE_ROOT`: checkpoints, run configs, graph stats, metrics, plots, diagnostics, comparison tables, manifests, and the DiFuMo coefficient caches. This notebook trains GAT/GATv2 only, so Colab memory is not shared with MLP training. If you have a real HCP DiFuMo-space FC matrix, upload it to Drive and set `HCP_FC_PATH`.

In [ ]:
from pathlib import Path
import os, time

DRIVE_ROOT = '/content/drive/MyDrive/neurovlm_difumo_gat'
RUN_ROOT = f'{DRIVE_ROOT}/runs'
RAW_COEFF_CACHE = f'{DRIVE_ROOT}/data/difumo512_pubmed_flatmap_coeffs.npz'
ALE_COEFF_CACHE = f'{DRIVE_ROOT}/data/difumo512_pubmed_ale_fwhm9_coeffs.npz'
COMPARISON_FILE = f'{DRIVE_ROOT}/runs/difumo_gat_comparison.csv'
# To resume a previous interrupted batch, set this to that folder name, e.g. '20260509_225451'.
RESUME_BATCH_NAME = None
RUN_BATCH_NAME = RESUME_BATCH_NAME or time.strftime('%Y%m%d_%H%M%S')
HCP_FC_PATH = ''  # e.g. f'{DRIVE_ROOT}/data/hcp_difumo512_fc.npy'

os.makedirs(RUN_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/data', exist_ok=True)

# GAT batches are much larger than MLP batches because each sample is a 512-node graph.
# Batch 128 OOMs on a 40GB A100 with hidden=128, heads=8, layers=3 and ~13K edges.
# Start safe; after one successful run, try 24/32 if VRAM allows.
GAT_BATCH_SIZE = 16
SHOW_COMMAND = False
FORCE_RERUN = False
STOP_ON_FAILURE = True

BASE = dict(
    epochs=80,
    batch_size=GAT_BATCH_SIZE,
    hidden=128,
    heads=8,
    layers=3,
    dropout=0.2,
    lr_gat=3e-4,
    lr_proj=3e-5,
    temperature=0.07,
    edge_threshold=95,
    early_stopping_patience=12,
    early_stopping_monitor='train_loss',
    early_stopping_min_delta=1e-4,
    val_interval=1,
    coeff_source='flatmap',
    coeff_cache_file=RAW_COEFF_CACHE,
    amp=True,
    umap=True,
)

CONFIGS = [
    dict(BASE, name='gatv2_raw_coactivation_h128_e95', model='gat', conv='gatv2', graph_type='coactivation'),
    dict(BASE, name='gatv2_ale_fwhm9_coactivation_h128_e95', model='gat', conv='gatv2', graph_type='coactivation', coeff_source='ale_coordinates', coeff_cache_file=ALE_COEFF_CACHE, ale_kernel_fwhm_mm=9),
    dict(BASE, name='gatv2_spatial_h128_e95', model='gat', conv='gatv2', graph_type='spatial', add_centroids=True),
    dict(BASE, name='gatv2_combined_h128_e95', model='gat', conv='gatv2', graph_type='combined', add_centroids=True),
    dict(BASE, name='gatv2_shuffled_control', model='gat', conv='gatv2', graph_type='shuffled'),
    dict(BASE, name='gatv2_no_edge_attr_control', model='gat', conv='gatv2', graph_type='coactivation', no_edge_attr=True),
    dict(BASE, name='gatv2_no_edge_control', model='gat', conv='gatv2', graph_type='no_edge'),
    dict(BASE, name='difumo_deepset_optional', model='deepset', graph_type='no_edge', add_centroids=True),
]
if HCP_FC_PATH:
    CONFIGS.extend([
        dict(BASE, name='gatv2_hcp_fc_h128_e95', model='gat', conv='gatv2', graph_type='hcp_fc', hcp_fc_path=HCP_FC_PATH),
        dict(BASE, name='gatv2_hcp_spatial_combined', model='gat', conv='gatv2', graph_type='combined', hcp_fc_path=HCP_FC_PATH, add_centroids=True),
    ])

# For a quick sanity pass, use CONFIGS[:2] and set epochs=3 in BASE above.
RUN_CONFIGS = CONFIGS
print('Drive root:', DRIVE_ROOT)
print('Comparison file:', COMPARISON_FILE)
print('Raw coefficient cache:', RAW_COEFF_CACHE)
print('ALE-smoothed coefficient cache:', ALE_COEFF_CACHE)
len(RUN_CONFIGS), [c['name'] for c in RUN_CONFIGS]

In [ ]:
import json, os, shlex, subprocess, sys, time

def cli_key(key):
    return '--' + key.replace('_', '-')

def run_config(cfg):
    name = cfg['name']
    run_dir = f'{RUN_ROOT}/{RUN_BATCH_NAME}/{name}'
    cmd = [
        sys.executable, 'experiments/train_difumo_gat_colab.py',
        '--run-dir', run_dir,
        '--checkpoint-dir', f'{run_dir}/checkpoints',
        '--comparison-file', COMPARISON_FILE,
    ]
    for key, value in cfg.items():
        if key == 'name' or value is None:
            continue
        if isinstance(value, bool):
            if value:
                cmd.append(cli_key(key))
            continue
        cmd.extend([cli_key(key), str(value)])
    os.makedirs(run_dir, exist_ok=True)
    log_path = f'{run_dir}/subprocess.log'
    done_path = f'{run_dir}/eval_results.json'
    ckpt_path = f'{run_dir}/checkpoints/best_difumo_gat.pt'
    print(f'\n=== {name} ===')
    print('Run dir:', run_dir)
    print('Log file:', log_path)
    if not FORCE_RERUN and os.path.exists(done_path) and os.path.exists(ckpt_path):
        print('Already completed; skipping. Set FORCE_RERUN=True to train again.')
        return {'name': name, 'status': 'skipped', 'run_dir': run_dir}
    if globals().get('SHOW_COMMAND', False):
        print('$', ' '.join(shlex.quote(x) for x in cmd))
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    started = time.time()
    tail = []
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            [cmd[0], '-u', *cmd[1:]],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
            tail.append(line.rstrip())
            tail = tail[-80:]
        return_code = proc.wait()
    elapsed_min = (time.time() - started) / 60
    if return_code != 0:
        print(f'\n{name} failed after {elapsed_min:.1f} min. Last log lines:')
        print('\n'.join(tail[-60:]))
        result = {'name': name, 'status': 'failed', 'run_dir': run_dir, 'log_path': log_path, 'return_code': return_code}
        if STOP_ON_FAILURE:
            raise RuntimeError(f'{name} failed with exit code {return_code}. Full log: {log_path}')
        return result
    print(f'{name} finished in {elapsed_min:.1f} min')
    manifest = {
        'name': name,
        'drive_run_dir': run_dir,
        'best_checkpoint': f'{run_dir}/checkpoints/best_difumo_gat.pt',
        'last_checkpoint': f'{run_dir}/checkpoints/last_difumo_gat.pt',
        'eval_results': f'{run_dir}/eval_results.json',
        'training_history': f'{run_dir}/training_history.json',
        'graph_stats': f'{run_dir}/graph_stats.json',
        'config': f'{run_dir}/config.json',
        'artifact_manifest': f'{run_dir}/artifacts_manifest.json',
        'comparison_file': COMPARISON_FILE,
        'coefficient_cache': cfg.get('coeff_cache_file'),
        'coefficient_source': cfg.get('coeff_source'),
    }
    with open(f'{run_dir}/colab_drive_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('Saved Drive artifacts under:', run_dir)
    return {'name': name, 'status': 'completed', 'run_dir': run_dir}

run_results = []
for cfg in RUN_CONFIGS:
    run_results.append(run_config(cfg))
run_results

In [ ]:
# Summarize results across controls.
import json, os
import pandas as pd
rows = []
for cfg in RUN_CONFIGS:
    run_dir = Path(RUN_ROOT) / RUN_BATCH_NAME / cfg['name']
    metrics_path = run_dir / 'eval_results.json'
    graph_path = run_dir / 'graph_stats.json'
    if not metrics_path.exists():
        continue
    metrics = json.loads(metrics_path.read_text())['test']
    graph = json.loads(graph_path.read_text()) if graph_path.exists() else {}
    rows.append({
        'run': cfg['name'],
        'model': cfg.get('model'),
        'graph': cfg.get('graph_type', 'none'),
        'auc': metrics['mean_auc'],
        'r@1': metrics['mean_recall@1'],
        'r@5': metrics['mean_recall@5'],
        'r@10': metrics['mean_recall@10'],
        'r@50': metrics['mean_recall@50'],
        'mrr': metrics['mean_mrr'],
        'median_rank': metrics['mean_median_rank'],
        'random_r@10': metrics['mean_random_recall@10'],
        'edges': graph.get('number_of_edges'),
        'avg_degree': graph.get('average_degree_directed'),
        'components': graph.get('connected_components'),
    })
summary = pd.DataFrame(rows).sort_values('auc', ascending=False)
display(summary)
summary_path = Path(RUN_ROOT) / RUN_BATCH_NAME / 'summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
print('Aggregate comparison CSV:', COMPARISON_FILE)

In [ ]:
# Display plots from the current best run.
from IPython.display import Image, display
if len(summary):
    best = summary.iloc[0]['run']
    print('Best run:', best)
    for png in ['training_curves.png', 'recall_curve.png', 'umap_diagnostics.png']:
        path = Path(RUN_ROOT) / RUN_BATCH_NAME / best / png
        if path.exists():
            display(Image(str(path)))
    for json_name in ['config.json', 'graph_stats.json']:
        path = Path(RUN_ROOT) / RUN_BATCH_NAME / best / json_name
        if path.exists():
            print('\n' + json_name)
            print(path.read_text()[:4000])

## Suggested Sweeps

After the first pass, sweep one axis at a time: `hidden in [64, 128, 256]`, `heads in [4, 8]`, `layers in [2, 3, 4]`, `batch_size in [8, 16, 24, 32]`, `lr_gat in [1e-4, 3e-4, 1e-5]`, `lr_proj in [1e-5, 3e-5]`, and `temperature in [0.03, 0.05, 0.07, 0.1]`.

Interpretation: if GAT beats the DiFuMo MLP, graph structure is helping. If only HCP FC helps, graph quality matters. If shuffled/no-edge GAT matches real-graph GAT, the graph is not adding much. If all DiFuMo methods trail atlas-free ALE3DCNN, the atlas bottleneck is probably limiting performance.